In [ ]:
# Import necessary libraries
import pandas as pd
from ast import literal_eval
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Read the test dataset and the models' predictions
test_set = pd.read_csv('final data/testing_dataset_triples.csv')
gpt4o_predictions = pd.read_csv('cte/gpt4o/results/gpt4o_results_triples.csv')
syntactic_parsing_predictions = pd.read_csv('cte/rule-based syntactic parsing/results/syntactic_parsing_results_triples.csv')
bert_predictions = pd.read_csv('cte/bert/results/bert_results_triples.csv')

In [ ]:
def calculate_f1_scores(matches, total_gold, total_pred):
    """
    Calculate precision, recall, and F1 score based on matches, total gold standard items, and total predicted items.

    """
    # Calculate precision, recall, and F1 score manually
    precision = sum(matches) / total_pred if total_pred else 0
    recall = sum(matches) / total_gold if total_gold else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) else 0
    return precision, recall, f1


def evaluate_triples(gold_df, pred_df):
    """
    Evaluate the predicted triples against the gold standard triples.
    
    """
    # Ensure tuples are correctly handled
    gold_df['triple'] = gold_df['triple'].apply(lambda x: x if isinstance(x, tuple) else literal_eval(x))
    pred_df['triple'] = pred_df['triple'].apply(lambda x: x if isinstance(x, tuple) else literal_eval(x))

    # Aggregate triples by sentence_id
    gold_triples = gold_df.groupby('sentence_id')['triple'].agg('first')
    pred_triples = pred_df.groupby('sentence_id')['triple'].agg('first')

    # Initialize match lists
    full_matches = []
    partial_matches = []
    subject_matches = []
    predicate_matches = []
    object_matches = []

    # Iterate over the indices to evaluate matches
    for idx in gold_triples.index:
        if idx in pred_triples:
            gold = gold_triples[idx]
            pred = pred_triples[idx]

            # Check for cases where either gold or predicted is completely empty
            if gold == ('', '', ''):
                if pred == ('', '', ''):
                    # Correct identification of no triple
                    full_matches.append(True)
                    partial_matches.append(True)
                    subject_matches.append(True)
                    predicate_matches.append(True)
                    object_matches.append(True)
                else:
                    # False positive: predicted a triple where there should be none
                    full_matches.append(False)
                    partial_matches.append(False)
                    subject_matches.append(False)
                    predicate_matches.append(False)
                    object_matches.append(False)
            elif pred == ('', '', ''):
                # False negative: failed to predict a triple that should exist
                full_matches.append(False)
                partial_matches.append(False)
                subject_matches.append(False)
                predicate_matches.append(False)
                object_matches.append(False)
            else:
                # Evaluate non-empty triples
                full_match = gold == pred
                full_matches.append(full_match)
                
                partial_match = False
                component_match = [False, False, False]
                for i, (g_comp, p_comp) in enumerate(zip(gold, pred)):
                    if g_comp == p_comp:
                        partial_match = True
                        component_match[i] = True

                partial_matches.append(partial_match)
                subject_matches.append(component_match[0])
                predicate_matches.append(component_match[1])
                object_matches.append(component_match[2])

    # Calculate precision, recall, F1 scores 
    full_match_precision, full_match_recall, full_match_f1 = calculate_f1_scores(full_matches, len(gold_triples), len(pred_triples))
    partial_match_precision, partial_match_recall, partial_match_f1 = calculate_f1_scores(partial_matches, len(gold_triples), len(pred_triples))
    subject_precision, subject_recall, subject_f1 = calculate_f1_scores(subject_matches, len(gold_triples), len(pred_triples))
    predicate_precision, predicate_recall, predicate_f1 = calculate_f1_scores(predicate_matches, len(gold_triples), len(pred_triples))
    object_precision, object_recall, object_f1 = calculate_f1_scores(object_matches, len(gold_triples), len(pred_triples))

    return {
        "full_match": 
        {"precision": full_match_precision, "recall": full_match_recall, "f1": full_match_f1},
        "partial_match": 
        {"precision": partial_match_precision, "recall": partial_match_recall, "f1": partial_match_f1},
        "subject_match": 
        {"precision": subject_precision, "recall": subject_recall, "f1": subject_f1},
        "predicate_match": 
        {"precision": predicate_precision, "recall": predicate_recall, "f1": predicate_f1},
        "object_match": 
        {"precision": object_precision, "recall": object_recall, "f1": object_f1}
    }


# Evaluate the triples comprehensively
evaluation_results_gpt4o = evaluate_triples(test_set, gpt4o_predictions)
evaluation_results_syntactic_parsing = evaluate_triples(test_set, syntactic_parsing_predictions)

print('The triples evaluation for the gpt4o model is:', evaluation_results_gpt4o)
print('The triples evaluation for the syntactic parsing model is:', evaluation_results_syntactic_parsing)


The triples evaluation for the gpt4o model is: {'full_match': {'precision': 0.35943775100401604, 'recall': 0.35943775100401604, 'f1': 0.359437751004016}, 'partial_match': {'precision': 0.642570281124498, 'recall': 0.642570281124498, 'f1': 0.642570281124498}, 'subject_match': {'precision': 0.5863453815261044, 'recall': 0.5863453815261044, 'f1': 0.5863453815261044}, 'predicate_match': {'precision': 0.4307228915662651, 'recall': 0.4307228915662651, 'f1': 0.4307228915662651}, 'object_match': {'precision': 0.4347389558232932, 'recall': 0.4347389558232932, 'f1': 0.4347389558232932}}
The triples evaluation for the syntactic parsing model is: {'full_match': {'precision': 0.001004016064257028, 'recall': 0.001004016064257028, 'f1': 0.001004016064257028}, 'partial_match': {'precision': 0.35441767068273095, 'recall': 0.35441767068273095, 'f1': 0.3544176706827309}, 'subject_match': {'precision': 0.26907630522088355, 'recall': 0.26907630522088355, 'f1': 0.26907630522088355}, 'predicate_match': {'pre

In [ ]:

def evaluate_triples_bert(data):
    """
    Evaluate the predicted triples against the gold standard triples for BERT predictions.
    
    """
    # Ensure tuples are correctly handled
    data['predicted_triple'] = data['predicted_triple'].apply(lambda x: x if isinstance(x, tuple) else literal_eval(x))
    data['gold_triple'] = data['gold_triple'].apply(lambda x: x if isinstance(x, tuple) else literal_eval(x))

    # Initialize match lists
    full_matches = []
    partial_matches = []
    subject_matches = []
    predicate_matches = []
    object_matches = []

    # Iterate over each row in the dataframe to evaluate matches
    for _, row in data.iterrows():
        gold = row['gold_triple']
        pred = row['predicted_triple']

        # Check for cases where either gold or predicted is completely empty
        if gold == ('', '', ''):
            if pred == ('', '', ''):
                # Correct identification of no triple
                full_matches.append(True)
                partial_matches.append(True)
                subject_matches.append(True)
                predicate_matches.append(True)
                object_matches.append(True)
            else:
                # False positive: predicted a triple where there should be none
                full_matches.append(False)
                partial_matches.append(False)
                subject_matches.append(False)
                predicate_matches.append(False)
                object_matches.append(False)
        elif pred == ('', '', ''):
            # False negative: failed to predict a triple that should exist
            full_matches.append(False)
            partial_matches.append(False)
            subject_matches.append(False)
            predicate_matches.append(False)
            object_matches.append(False)
        else:
            # Evaluate non-empty triples
            full_match = gold == pred
            full_matches.append(full_match)
            
            partial_match = False
            component_match = [False, False, False]
            for i, (g_comp, p_comp) in enumerate(zip(gold, pred)):
                if g_comp == p_comp:
                    partial_match = True
                    component_match[i] = True

            partial_matches.append(partial_match)
            subject_matches.append(component_match[0])
            predicate_matches.append(component_match[1])
            object_matches.append(component_match[2])

    # Calculate precision, recall, F1 scores 
    full_match_precision, full_match_recall, full_match_f1 = calculate_f1_scores(full_matches, len(data), len(data))
    partial_match_precision, partial_match_recall, partial_match_f1 = calculate_f1_scores(partial_matches, len(data), len(data))
    subject_precision, subject_recall, subject_f1 = calculate_f1_scores(subject_matches, len(data), len(data))
    predicate_precision, predicate_recall, predicate_f1 = calculate_f1_scores(predicate_matches, len(data), len(data))
    object_precision, object_recall, object_f1 = calculate_f1_scores(object_matches, len(data), len(data))

    return {
        "full_match": 
        {"precision": full_match_precision, "recall": full_match_recall, "f1": full_match_f1},
        "partial_match": 
        {"precision": partial_match_precision, "recall": partial_match_recall, "f1": partial_match_f1},
        "subject_match": 
        {"precision": subject_precision, "recall": subject_recall, "f1": subject_f1},
        "predicate_match": 
        {"precision": predicate_precision, "recall": predicate_recall, "f1": predicate_f1},
        "object_match": 
        {"precision": object_precision, "recall": object_recall, "f1": object_f1}
    }


evaluation_results = evaluate_triples_bert(bert_predictions)
print(evaluation_results)